# Lab: Hồi quy Logistic (Logistic Regression)

## 1. Vì sao không dùng Linear Regression cho phân loại?

Linear Regression cho output là một số thực bất kỳ — có thể âm, có thể >1. Trong phân loại nhị phân, ta muốn output ∈ [0, 1] hiểu là *xác suất* thuộc lớp 1.

Giải pháp: **đưa $w^T x + b$ qua hàm sigmoid** để ép về [0, 1]:
$$
\hat{p}(x) = \sigma(w^T x + b) = \frac{1}{1 + e^{-(w^T x + b)}}
$$

Đây là **Logistic Regression** — nghe tên thì giống regression nhưng thực chất là **classifier**.

## 2. Hàm Sigmoid

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Tính chất:
- $\sigma(0) = 0.5$.
- $\sigma(z) \to 1$ khi $z \to +\infty$, $\sigma(z) \to 0$ khi $z \to -\infty$.
- Đạo hàm rất đẹp: $\sigma'(z) = \sigma(z)(1 - \sigma(z))$.

Sau khi có $\hat{p}$, dự đoán nhãn theo ngưỡng (mặc định 0.5):
$$\hat{y} = \begin{cases}1 & \hat{p} \ge 0.5 \\ 0 & \hat{p} < 0.5\end{cases}$$

## 3. Hàm mất mát: Binary Cross-Entropy

Vì sao **không** dùng MSE? MSE + sigmoid → loss landscape có nhiều local minima → khó train. Hơn nữa, gradient gần 0 ở vùng $\hat{p}$ gần 0 hoặc 1 (sigmoid bão hoà).

Logistic Regression dùng **Binary Cross-Entropy (BCE)**:
$$
L = -\frac{1}{N}\sum_{i=1}^{N}\Big[y_i \log \hat{p}_i + (1 - y_i) \log(1 - \hat{p}_i)\Big]
$$

Đọc bằng lời:
- Nếu $y_i = 1$: muốn $\hat{p}_i$ gần 1 → $-\log \hat{p}_i$ nhỏ.
- Nếu $y_i = 0$: muốn $\hat{p}_i$ gần 0 → $-\log(1 - \hat{p}_i)$ nhỏ.

BCE convex theo $w$ → Gradient Descent đảm bảo về cực tiểu toàn cục.

## 4. Một công thức gradient cực đẹp

Khi tổ hợp Sigmoid + BCE, gradient có dạng đơn giản:
$$\frac{\partial L}{\partial w_j} = \frac{1}{N}\sum_{i=1}^{N}(\hat{p}_i - y_i) \cdot x_{i,j}$$

Nghĩa là: gradient = (sai số) × (input). Đây cũng chính là gradient của Linear Regression với MSE — chỉ khác $\hat{p}$ là sigmoid thay vì linear. Đây là một *vẻ đẹp toán học* sâu sắc đứng đằng sau cả deep learning.

---

## 0. Thử dùng Linear Regression cho phân loại — hỏng ở đâu?

Cách tự nhiên nhất khi mới học: gán nhãn 0/1 rồi chạy thẳng `LinearRegression`, ai lớn hơn 0.5 thì cho là lớp 1. Nghe hợp lý. Nhưng nó hỏng ở ba chỗ:

![Vì sao không dùng Linear Regression cho phân loại](images/01_vi_sao_khong_dung_linear.png)

*Trái: đường thẳng chui xuống dưới 0 và vọt lên trên 1 (vùng tô đỏ) — những giá trị đó **không thể** đọc là xác suất. Giữa: thêm 3 điểm lớp 1 ở rất xa, đường thẳng bị kéo nghiêng và ngưỡng dịch hẳn sang phải, phân loại sai một loạt điểm lớp 1 ở giữa. Phải: Logistic Regression trên **đúng bộ dữ liệu có outlier đó** gần như không suy suyển.*

| Vấn đề | Linear Regression | Logistic Regression |
|---|---|---|
| Output ngoài $[0,1]$ | Có — vô nghĩa khi cần xác suất | Không bao giờ, nhờ sigmoid |
| Điểm ở xa kéo lệch ranh giới | Rất mạnh (MSE phạt bình phương) | Yếu (điểm đã đúng chắc chắn thì gradient $\approx 0$) |
| Giả định về nhiễu | Gaussian — sai với dữ liệu nhị phân | Bernoulli — đúng với dữ liệu nhị phân |

Cái thứ ba là gốc rễ. Nhãn 0/1 tuân theo phân phối **Bernoulli**, không phải Gaussian. Linear Regression giả định sai ngay từ đầu; Logistic Regression giả định đúng. Mọi thứ khác chỉ là hệ quả.

## 2b. Sigmoid: đọc kỹ từng tính chất

![Hàm sigmoid và đạo hàm](images/02_ham_sigmoid.png)

*Trái: sigmoid ép trục số thực vô hạn về khoảng $(0,1)$; hai đầu là **vùng bão hoà** — thay đổi $z$ rất nhiều mà $\hat{p}$ gần như đứng yên. Phải: đạo hàm $\sigma'(z)=\sigma(z)(1-\sigma(z))$ đạt cực đại chỉ **0.25** tại $z=0$ và tắt rất nhanh về hai phía.*

Bốn tính chất phải thuộc:

1. **Đối xứng:** $\sigma(-z) = 1 - \sigma(z)$. Nên $\hat{p}(y=0) = \sigma(-z)$ — chỉ cần một hàm cho cả hai lớp.
2. **Đạo hàm gọn:** $\sigma'(z) = \sigma(z)\big(1 - \sigma(z)\big)$. Tính $\sigma$ một lần là có luôn đạo hàm — cực rẻ khi backprop.
3. **Bão hoà:** khi $|z| > 5$ thì $\sigma' < 0.007$. Gradient gần như bằng 0 → tham số gần như không cập nhật nữa.
4. **Hàm ngược là logit:** $\sigma^{-1}(p) = \ln\frac{p}{1-p}$.

Tính chất 3 chính là mầm mống của **vanishing gradient** — thủ phạm khiến mạng neural sâu dùng sigmoid không train được, và là lý do ReLU ra đời (ta sẽ gặp lại ở Lab 09).

> ⚠️ **Bẫy số học:** `1/(1+np.exp(-z))` bị tràn khi `z` rất âm (`exp(1000) = inf`). Trong lab ta dùng `np.clip(z, -500, 500)`. Thư viện thật dùng công thức ổn định theo dấu của $z$, hoặc `scipy.special.expit`.

## 2c. Odds, logit và cách ĐỌC hệ số $w_j$

![Odds và logit](images/03_odds_va_logit.png)

Đây là phần bị bỏ qua nhiều nhất nhưng lại là thứ khiến Logistic Regression được ưa chuộng trong y học và tài chính: **hệ số của nó diễn giải được**.

- **Odds** (tỷ lệ cược): $\text{odds} = \dfrac{p}{1-p}$. Ví dụ $p = 0.8 \Rightarrow$ odds $= 4$, nghĩa là "khả năng xảy ra gấp 4 lần không xảy ra".
- **Logit** = $\ln(\text{odds})$, kéo $[0,1]$ ra thành $(-\infty, +\infty)$.

Viết ngược lại mô hình:
$$
\ln \frac{p}{1-p} = w^Tx + b
$$

Đọc thành lời: **Logistic Regression là hồi quy tuyến tính trên thang log-odds.** Từ đó suy ra cách đọc hệ số:

> Khi $x_j$ tăng thêm **1 đơn vị** (các biến khác giữ nguyên), **odds nhân lên $e^{w_j}$ lần**. Đại lượng $e^{w_j}$ gọi là **Odds Ratio (OR)**.

| $w_j$ | $e^{w_j}$ | Diễn giải |
|---|---|---|
| $0$ | 1.00 | Feature không ảnh hưởng |
| $0.69$ | 2.00 | Tăng 1 đơn vị → odds **gấp đôi** |
| $-0.69$ | 0.50 | Tăng 1 đơn vị → odds **giảm một nửa** |
| $1.10$ | 3.00 | Tăng 1 đơn vị → odds gấp 3 |

Ví dụ báo cáo y khoa: "Hút thuốc có $w = 1.4$, tức $OR = e^{1.4} \approx 4.05$ — người hút thuốc có odds mắc bệnh cao gấp **4 lần** người không hút, sau khi đã hiệu chỉnh các yếu tố khác."

> ⚠️ Hai cảnh báo: (1) OR nói về **odds**, không phải xác suất — chỉ khi bệnh hiếm ($p$ nhỏ) thì OR mới xấp xỉ risk ratio. (2) Nếu bạn đã `StandardScaler`, hệ số ứng với "tăng 1 **độ lệch chuẩn**", không phải 1 đơn vị gốc. Muốn diễn giải theo đơn vị gốc thì chia lại cho $\sigma_j$.

## 3b. BCE đến từ đâu? — Suy ra từ hợp lý cực đại

BCE không phải công thức trên trời rơi xuống. Với nhãn nhị phân, mỗi quan sát là một phép thử **Bernoulli** có xác suất thành công $\hat{p}_i$:
$$
P(y_i \mid x_i) = \hat{p}_i^{\,y_i}\,(1 - \hat{p}_i)^{\,1 - y_i}
$$
(Kiểm tra nhanh: $y_i=1$ cho $\hat{p}_i$; $y_i=0$ cho $1-\hat{p}_i$. Một công thức gói gọn cả hai trường hợp.)

Giả sử các mẫu độc lập, hàm hợp lý của cả tập là tích, và log-likelihood là tổng:
$$
\log \mathcal{L}(w,b) = \sum_{i=1}^{N}\Big[y_i \log \hat{p}_i + (1-y_i)\log(1-\hat{p}_i)\Big]
$$

**Cực đại log-likelihood** $\iff$ **cực tiểu** đúng đại lượng đó với dấu trừ và chia $N$ — chính là BCE. Vậy: *tối thiểu BCE = ước lượng hợp lý cực đại cho mô hình Bernoulli.* Cùng logic đã dẫn MSE ra từ giả định Gaussian ở Lab 01.

![BCE so với MSE](images/04_bce_vs_mse.png)

*Trái: BCE phạt **vô hạn** khi model sai mà lại tự tin — nói $\hat{p}=0.01$ trong khi $y=1$ thì loss $=4.6$; nói $\hat{p}=0.001$ thì loss $=6.9$. Giữa: mặt cắt loss theo tham số — BCE là **hàm lồi**, chỉ một đáy. Phải: MSE ghép với sigmoid tạo ra hai **cao nguyên phẳng** hai bên; ở đó gradient $\approx 0$ nên Gradient Descent đứng im mãi mãi.*

### Vì sao MSE + sigmoid lại tệ đến thế

Tính gradient của MSE khi có sigmoid ở giữa:
$$
\frac{\partial}{\partial w_j}\,\frac{1}{N}\sum_i (\hat{p}_i - y_i)^2 = \frac{2}{N}\sum_i (\hat{p}_i - y_i)\;\underbrace{\sigma'(z_i)}_{\text{thủ phạm}}\;x_{ij}
$$

Thừa số $\sigma'(z_i)$ tiến về 0 ở vùng bão hoà. Nghĩa là: **một mẫu bị dự đoán sai bét (ví dụ $y=1$ nhưng $\hat{p}=0.001$) lại tạo ra gradient gần bằng 0** — model sai nặng nhưng không học được gì. Đây đúng là điều tệ nhất có thể xảy ra.

Với BCE, phép màu xảy ra: $\sigma'$ bị **triệt tiêu** trong lúc rút gọn, còn lại
$$
\frac{\partial L}{\partial w_j} = \frac{1}{N}\sum_i (\hat{p}_i - y_i)\,x_{ij}
$$
Sai càng nhiều thì gradient càng lớn — đúng như trực giác mong đợi.

### Chứng minh BCE lồi (phác thảo)

Với một mẫu, viết BCE theo $z$: $\ell(z) = \log(1+e^{z}) - yz$. Đạo hàm bậc hai:
$$
\ell''(z) = \sigma(z)\big(1-\sigma(z)\big) > 0 \quad \forall z
$$
Vậy $\ell$ lồi chặt theo $z$; $z = w^Tx+b$ là hàm affine của $(w,b)$; **hợp của hàm lồi với hàm affine vẫn lồi**; tổng các hàm lồi vẫn lồi. Kết luận: BCE lồi theo $(w,b)$ → Gradient Descent không thể kẹt ở cực tiểu địa phương.

> Lưu ý tinh tế: lồi **không** đồng nghĩa với "có nghiệm hữu hạn". Nếu dữ liệu tách được tuyến tính hoàn hảo, BCE giảm mãi khi $\|w\| \to \infty$ mà không bao giờ đạt cực tiểu. Đó là lý do sklearn **luôn bật regularization mặc định** — xem mục 6b bên dưới.

## 4b. Suy ra gradient bằng chain rule — và hình học của nó

Đặt $z_i = w^Tx_i + b$, $\hat{p}_i = \sigma(z_i)$. Đi ngược từ loss về tham số:

$$
\frac{\partial \ell_i}{\partial \hat{p}_i}
= -\frac{y_i}{\hat{p}_i} + \frac{1-y_i}{1-\hat{p}_i}
= \frac{\hat{p}_i - y_i}{\hat{p}_i(1-\hat{p}_i)}
$$

$$
\frac{\partial \hat{p}_i}{\partial z_i} = \sigma'(z_i) = \hat{p}_i(1-\hat{p}_i)
$$

Nhân hai vế — **mẫu số và tử số triệt tiêu hoàn toàn**:
$$
\frac{\partial \ell_i}{\partial z_i} = \frac{\hat{p}_i - y_i}{\cancel{\hat{p}_i(1-\hat{p}_i)}} \cdot \cancel{\hat{p}_i(1-\hat{p}_i)} = \hat{p}_i - y_i
$$

Cuối cùng, vì $\partial z_i/\partial w_j = x_{ij}$ và $\partial z_i/\partial b = 1$:
$$
\boxed{\;\nabla_w L = \frac{1}{N}X^T(\hat{p} - y), \qquad \frac{\partial L}{\partial b} = \frac{1}{N}\sum_i(\hat{p}_i - y_i)\;}
$$

Đây **đúng bằng** công thức gradient của hồi quy tuyến tính với MSE, chỉ khác $\hat{p}$ thay cho $\hat{y}$. Không phải trùng hợp: cả hai đều thuộc họ **Generalized Linear Model**, và với mọi GLM dùng "canonical link" thì gradient luôn có dạng $X^T(\text{dự đoán} - \text{thực tế})$. Cũng chính công thức này chạy trong mọi tầng cuối của mạng neural phân loại.

**Khác biệt quan trọng so với Lab 01:** hồi quy tuyến tính có nghiệm đóng (Normal Equation), còn Logistic Regression **không có**. Phương trình $X^T(\sigma(X\theta) - y) = 0$ phi tuyến theo $\theta$ nên buộc phải giải lặp — bằng Gradient Descent, hoặc bằng Newton (còn gọi là IRLS), hoặc L-BFGS như sklearn dùng mặc định.

![Hình học của Logistic Regression](images/05_bien_quyet_dinh.png)

*Trái: ranh giới quyết định là nơi $\hat{p}=0.5$, tức $w^Tx+b=0$ — **luôn là một đường thẳng** (siêu phẳng trong nhiều chiều). Các đường cong màu khác là đường đồng mức xác suất, luôn song song với ranh giới. Mũi tên xanh lá là vector $w$: nó vuông góc với ranh giới và chỉ về phía lớp 1. Phải: chiếu toàn bộ dữ liệu lên hướng $w$ thì bài toán thu về đúng một đường sigmoid 1 chiều — $\|w\|$ quyết định đường đó dốc hay thoải.*

Hai hệ quả cần nhớ:
1. **Ranh giới luôn thẳng.** Muốn cong thì phải tự tạo feature cong (polynomial, spline) hoặc đổi sang model khác.
2. **$\|w\|$ điều khiển độ tự tin**, còn hướng của $w$ điều khiển vị trí ranh giới. Regularization tác động lên $\|w\|$ → tác động lên **độ tự tin** chứ không chỉ vị trí.

## 5. Mở rộng cho Multiclass: Softmax Regression

Với $C$ lớp, ta dùng:
$$\hat{p}_c(x) = \frac{e^{w_c^T x + b_c}}{\sum_{k=1}^{C} e^{w_k^T x + b_k}}$$

Đây là **softmax** — hàm xác suất nhiều chiều. Loss tương ứng là **categorical cross-entropy**:
$$L = -\frac{1}{N}\sum_{i=1}^{N}\sum_{c=1}^{C} y_{i,c} \log \hat{p}_{i,c}$$

Trong sklearn, `LogisticRegression()` **chính là** Softmax Regression khi dữ liệu có nhiều hơn 2 lớp — không cần (và từ phiên bản 1.7 là không còn) tham số `multi_class`.

### Softmax: hai cách làm nhiều lớp và vài chi tiết quan trọng

![Softmax và phân loại đa lớp](images/08_softmax_da_lop.png)

*Trái: ba xác suất softmax luôn cộng lại đúng bằng 1 — chúng "cạnh tranh" nhau. Giữa và phải: `multinomial` giải một bài toán tối ưu chung cho cả 3 lớp, còn `one-vs-rest` huấn luyện 3 bộ nhị phân độc lập rồi so điểm. Trên dữ liệu dễ, hai cách gần như trùng nhau; trên dữ liệu chồng lấn, multinomial cho xác suất hiệu chỉnh tốt hơn.*

| | Multinomial (Softmax) | One-vs-Rest (OvR) |
|---|---|---|
| Số bài toán tối ưu | 1 (chung) | $C$ bài độc lập |
| Tổng xác suất | Đúng bằng 1 theo thiết kế | Phải chuẩn hoá lại thủ công |
| Dữ liệu mất cân bằng | Xử lý tự nhiên | Mỗi bộ nhị phân đều bị lệch "1 chọi tất cả" |
| Đa **nhãn** (một mẫu nhiều nhãn) | Không dùng được | **Dùng được** — đây là ưu thế thật của OvR |
| Trong sklearn | Mặc định của `LogisticRegression` | `OneVsRestClassifier(LogisticRegression())` |

> ⚠️ **Cập nhật API:** tham số `multi_class='multinomial'` đã bị **loại bỏ** khỏi `LogisticRegression` từ scikit-learn 1.7. Hiện tại `LogisticRegression` **mặc định luôn là multinomial** khi có nhiều hơn 2 lớp — cứ gọi `LogisticRegression()` là đủ. Nếu thật sự cần OvR, hãy bọc bằng `OneVsRestClassifier`. Code cũ có `multi_class=...` sẽ báo `TypeError` trên sklearn mới.

**Hai tính chất của softmax cần biết:**

1. **Bất biến với phép tịnh tiến:** $\text{softmax}(z + c) = \text{softmax}(z)$ với mọi hằng số $c$. Hệ quả: mô hình **thừa tham số** — cộng cùng một vector vào mọi $w_c$ thì kết quả không đổi. Vì vậy bài nhị phân chỉ cần 1 bộ trọng số chứ không phải 2.

2. **Log-sum-exp trick:** tính trực tiếp $e^{z_c}$ dễ tràn số khi $z_c$ lớn. Mọi thư viện đều trừ đi giá trị lớn nhất trước:
$$
\text{softmax}(z)_c = \frac{e^{z_c - \max_k z_k}}{\sum_j e^{z_j - \max_k z_k}}
$$
Kết quả toán học y hệt nhưng không bao giờ tràn. Đây cũng chính là lý do PyTorch bắt bạn đưa **logits** (chưa softmax) vào `nn.CrossEntropyLoss` — nó gộp softmax và log lại để dùng được trick này. Ta sẽ gặp lại ở Lab 09.

# THỰC HÀNH 1: Logistic Regression từ scratch trên data 2D

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, load_breast_cancer, load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_curve, auc)

np.random.seed(42)

# Sinh dữ liệu 2D có thể phân loại tuyến tính được
X, y = make_classification(n_samples=200, n_features=2, n_redundant=0,
                            n_informative=2, n_clusters_per_class=1,
                            random_state=42)

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='k', s=30)
plt.title('Dữ liệu 2D, 2 lớp'); plt.grid(alpha=0.3); plt.show()

In [ ]:
# Cài Logistic Regression bằng Gradient Descent thủ công
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

class MyLogReg:
    def __init__(self, lr=0.1, n_iter=2000):
        self.lr = lr
        self.n_iter = n_iter

    def fit(self, X, y):
        N, d = X.shape
        self.w = np.zeros(d)
        self.b = 0.0
        self.loss_hist = []
        for _ in range(self.n_iter):
            z = X @ self.w + self.b
            p = sigmoid(z)
            # BCE loss (cộng epsilon tránh log 0)
            loss = -np.mean(y * np.log(p + 1e-9) + (1 - y) * np.log(1 - p + 1e-9))
            self.loss_hist.append(loss)
            # gradient
            dw = (X.T @ (p - y)) / N
            db = (p - y).mean()
            self.w -= self.lr * dw
            self.b -= self.lr * db
        return self

    def predict_proba(self, X):
        return sigmoid(X @ self.w + self.b)

    def predict(self, X, thresh=0.5):
        return (self.predict_proba(X) >= thresh).astype(int)

mine = MyLogReg(lr=0.1, n_iter=2000).fit(X, y)
print(f'My LR     accuracy: {(mine.predict(X) == y).mean()*100:.2f}%')
print(f'w = {mine.w}, b = {mine.b:.4f}')

skl = LogisticRegression(C=1e10).fit(X, y)   # C lớn = ít regularization, khớp model thuần
print(f'sklearn   accuracy: {skl.score(X, y)*100:.2f}%')
print(f'w = {skl.coef_[0]}, b = {skl.intercept_[0]:.4f}')

In [ ]:
# Vẽ decision boundary
xx, yy = np.meshgrid(np.linspace(X[:, 0].min()-1, X[:, 0].max()+1, 200),
                      np.linspace(X[:, 1].min()-1, X[:, 1].max()+1, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
proba = mine.predict_proba(grid).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
cs = axes[0].contourf(xx, yy, proba, levels=20, cmap='coolwarm', alpha=0.7)
axes[0].contour(xx, yy, proba, levels=[0.5], colors='black', linestyles='--')
axes[0].scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='white', s=30)
plt.colorbar(cs, ax=axes[0], label='P(y=1)')
axes[0].set_title('Xác suất + boundary 0.5'); axes[0].grid(alpha=0.3)

axes[1].plot(mine.loss_hist); axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('BCE loss')
axes[1].set_title('Loss giảm dần qua GD'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6b. Regularization trong Logistic Regression — và bài toán "tách hoàn toàn"

![Ảnh hưởng của tham số C](images/06_regularization_C.png)

*Ba panel trái: cùng dữ liệu, chỉ đổi C. **C nhỏ** = phạt mạnh = $\|w\|$ nhỏ = vùng chuyển tiếp màu rộng và mờ (model dè dặt). **C lớn** = phạt yếu = $\|w\|$ lớn = chuyển tiếp gắt như bậc thang (model rất tự tin). Panel phải: với dữ liệu **tách tuyến tính hoàn toàn**, càng bỏ regularization thì $\|w\|$ càng phóng ra vô cực — không hề hội tụ.*

**Hiện tượng perfect separation.** Nếu tồn tại một siêu phẳng tách sạch hai lớp, thì nhân $w$ và $b$ lên gấp đôi sẽ khiến mọi $\hat{p}$ tiến gần 0 hoặc 1 hơn, tức BCE giảm tiếp. Cứ thế mãi: **cực tiểu nằm ở vô cực**, ước lượng hợp lý cực đại không tồn tại. Triệu chứng thực tế: hệ số khổng lồ (hàng trăm, hàng nghìn), cảnh báo `ConvergenceWarning`, và model tự tin 100% một cách vô lý.

Regularization chữa dứt điểm: thêm $\frac{1}{2C}\|w\|^2$ thì hàm mục tiêu trở nên **lồi chặt và có nghiệm hữu hạn**. Đây là lý do `LogisticRegression` của sklearn **luôn bật L2 mặc định với `C=1.0`** — khác hẳn `LinearRegression` vốn không phạt gì.

$$
L = \underbrace{-\frac{1}{N}\sum_i\big[y_i\log\hat{p}_i + (1-y_i)\log(1-\hat{p}_i)\big]}_{\text{BCE}} \;+\; \underbrace{\frac{1}{2C}\|w\|_2^2}_{\text{phạt}}
$$

> ⚠️ **Nhớ kỹ: $C = 1/\alpha$.** Ngược chiều với `alpha` của Ridge/Lasso ở Lab 01. **C nhỏ = phạt mạnh.** Đây là chỗ sinh viên nhầm nhiều nhất khi làm bài tập sweep tham số.

### Chọn penalty và solver cho khớp nhau

| Penalty | Tác dụng | Solver hỗ trợ |
|---|---|---|
| `'l2'` (mặc định) | Co hệ số, ổn định, giữ mọi feature | `lbfgs`, `newton-cg`, `newton-cholesky`, `sag`, `saga`, `liblinear` |
| `'l1'` | Đẩy hệ số về **đúng 0** → chọn feature | `liblinear`, `saga` |
| `'elasticnet'` | Lai L1+L2, cần thêm `l1_ratio` | **Chỉ** `saga` |
| `None` | Không phạt — chỉ dùng khi muốn MLE thuần | `lbfgs`, `newton-cg`, `sag`, `saga` |

| Solver | Hợp với | Ghi chú |
|---|---|---|
| `lbfgs` | Mặc định, dataset vừa | Nhanh, chỉ L2 hoặc không phạt |
| `liblinear` | Dataset nhỏ, cần L1 | Chỉ làm OvR khi nhiều lớp |
| `saga` | Dataset **lớn**, cần L1/ElasticNet | Nên scale feature trước, nếu không hội tụ rất chậm |
| `newton-cholesky` | $N \gg n$ (nhiều mẫu, ít feature) | Rất nhanh trong trường hợp đó |

**Gặp `ConvergenceWarning` thì làm gì?** Theo thứ tự: (1) `StandardScaler` dữ liệu — thường là đủ; (2) tăng `max_iter` lên 1000–5000; (3) giảm `C` (phạt mạnh hơn); (4) đổi solver. Đừng bao giờ chỉ tăng `max_iter` rồi bỏ qua cảnh báo mà không scale.

> ⚠️ **Không phạt hệ số chệch $b$.** sklearn không đưa `intercept_` vào penalty — đúng, vì phạt $b$ tương đương ép ranh giới phải đi gần gốc toạ độ, một ràng buộc hoàn toàn vô nghĩa.

# THỰC HÀNH 2: Phân loại ung thư trên Breast Cancer

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# QUAN TRỌNG: scale feature trước Logistic Regression khi có regularization
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)
y_pred  = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)[:, 1]

print(classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
# ROC + AUC
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc_val = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
axes[0].plot(fpr, tpr, linewidth=2, label=f'AUC = {auc_val:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', label='Đoán mò')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Top 10 feature importance
imp = pd.Series(np.abs(model.coef_[0]), index=data.feature_names).nlargest(10)
imp.plot.barh(ax=axes[1])
axes[1].set_xlabel('|hệ số|'); axes[1].set_title('Top 10 feature quan trọng nhất')
plt.tight_layout(); plt.show()

## 6. Threshold tuning

Mặc định predict ở ngưỡng 0.5. Trong y học, có khi muốn giảm bỏ sót (recall cao hơn) — chấp nhận nhiều báo động giả. Đổi ngưỡng để cân bằng precision/recall theo nhu cầu.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.1, 0.95, 0.05)
precs, recs, f1s = [], [], []
for t in thresholds:
    pred = (y_proba >= t).astype(int)
    precs.append(precision_score(y_test, pred, zero_division=0))
    recs.append(recall_score(y_test, pred))
    f1s.append(f1_score(y_test, pred))

plt.figure(figsize=(8, 4.5))
plt.plot(thresholds, precs, 'o-', label='Precision')
plt.plot(thresholds, recs, 's-', label='Recall')
plt.plot(thresholds, f1s, '^-', label='F1')
best_t = thresholds[np.argmax(f1s)]
plt.axvline(best_t, color='red', linestyle='--', label=f'Best F1 ở {best_t:.2f}')
plt.xlabel('Ngưỡng quyết định'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Precision / Recall / F1 thay đổi theo ngưỡng')
plt.show()

### Muốn ranh giới cong? Thêm feature, đừng đổi model vội

![Logistic Regression với polynomial features](images/07_polynomial_logistic.png)

*Cùng một `LogisticRegression`, chỉ khác ở chỗ ta nạp thêm $x_1^2, x_1x_2, x_2^2, \dots$ vào input. `degree=1` chỉ vẽ được đường thẳng; `degree=3` bám đẹp hai vành trăng; `degree=12` bắt đầu uốn éo theo từng hạt nhiễu — overfit.*

Đây là bài học lặp lại của Lab 01: **"tuyến tính" nói về tuyến tính theo THAM SỐ, không phải theo dữ liệu.** Model vẫn là $\sigma(w^T\phi(x)+b)$ — vẫn tuyến tính theo $w$, vẫn lồi, vẫn train bằng đúng thuật toán cũ. Chỉ có $\phi(x)$ được làm giàu thêm.

| Cách tạo ranh giới cong | Ưu | Nhược |
|---|---|---|
| `PolynomialFeatures` | Đơn giản, giữ tính giải thích | Số feature nổ theo $\binom{n+d}{d}$; cần regularization mạnh |
| Spline / binning | Linh hoạt cục bộ, ổn định hơn đa thức bậc cao | Phải chọn số nút |
| **Kernel SVM** (Lab 06) | Không cần tạo feature tường minh | Mất tính giải thích, chậm khi $N$ lớn |
| **MLP** (Lab 09) | Tự học biến đổi phi tuyến | Cần nhiều dữ liệu, khó giải thích |

**Quy trình thực dụng:** luôn chạy Logistic Regression thuần trước để có **baseline**. Nếu nó đã đủ tốt thì dừng — bạn được kèm luôn một mô hình giải thích được. Chỉ leo thang độ phức tạp khi baseline thật sự không đạt yêu cầu.

# THỰC HÀNH 3: Multiclass Logistic (Softmax) trên Iris

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Từ scikit-learn 1.7, tham số multi_class đã bị loại bỏ:
# LogisticRegression tự động dùng softmax (multinomial) khi có > 2 lớp.
# Muốn one-vs-rest thì bọc: OneVsRestClassifier(LogisticRegression())
model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)

print(f'Test accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')
print(classification_report(y_test, y_pred, target_names=iris.target_names))

### Chọn ngưỡng: phần quyết định giá trị thực tế của model

![Ngưỡng quyết định](images/09_nguong_quyet_dinh.png)

*Trái: model xuất ra một **phân phối điểm số**; ngưỡng chỉ là đường kẻ dọc mà ta tự đặt. Giữa: precision tăng còn recall giảm khi đẩy ngưỡng lên — ở bộ dữ liệu này F1 đạt cực đại **không phải** tại 0.5. Phải: mỗi ngưỡng tương ứng đúng một điểm trên đường ROC.*

Điểm mấu chốt: **đổi ngưỡng KHÔNG cần train lại.** Model đã học xong; ngưỡng chỉ là cách ta *đọc* xác suất. Vì vậy hãy tách bạch hai việc:
1. Train model tốt nhất có thể (đo bằng AUC / average precision — các chỉ số **không phụ thuộc ngưỡng**).
2. Chọn ngưỡng theo nhu cầu nghiệp vụ (đo bằng precision/recall/F1 tại ngưỡng đó).

### Ngưỡng tối ưu khi biết chi phí

Nếu bỏ sót một ca dương tính tốn $c_{FN}$ và báo động giả tốn $c_{FP}$, thì quyết định "gọi là dương" có lợi khi kỳ vọng chi phí thấp hơn:
$$
\hat{p}\,\cdot\,0 + (1-\hat{p})\,c_{FP} \;<\; \hat{p}\,c_{FN} + (1-\hat{p})\cdot 0
\quad\Longrightarrow\quad
\boxed{\;\hat{p} > \frac{c_{FP}}{c_{FP} + c_{FN}}\;}
$$

Ngưỡng 0.5 chỉ đúng khi $c_{FP} = c_{FN}$ — **hiếm khi đúng trong thực tế**. Ví dụ: bỏ sót ung thư đắt gấp 20 lần một lần xét nghiệm thừa → ngưỡng tối ưu $= \frac{1}{1+20} \approx 0.048$, tức nên báo dương ngay khi $\hat{p} > 5\%$.

| Bài toán | Sai lầm đắt hơn | Ngưỡng nên | Chỉ số ưu tiên |
|---|---|---|---|
| Sàng lọc ung thư | Bỏ sót (FN) | **Thấp** (0.05–0.2) | Recall |
| Lọc spam | Chặn nhầm mail thật (FP) | **Cao** (0.7–0.9) | Precision |
| Duyệt vay | Tuỳ khẩu vị rủi ro | Theo ma trận chi phí | Lợi nhuận kỳ vọng |
| Xếp hạng gợi ý | Không quyết định nhị phân | Không cần ngưỡng | AUC, NDCG |

> ⚠️ **Bẫy:** chọn ngưỡng trên **tập test** rồi báo cáo kết quả trên chính tập test đó là gian lận (giống hệt việc tune hyperparameter trên test). Hãy chọn ngưỡng trên tập validation hoặc bằng cross-validation.

Chi tiết đầy đủ về precision, recall, ROC, PR curve và calibration nằm ở **Lab 07 — Đánh giá mô hình**.

## Tổng kết

1. Logistic Regression = Linear Regression + Sigmoid + BCE → cho xác suất phân loại nhị phân.
2. **Phải scale feature** khi có regularization (mặc định sklearn dùng L2).
3. **Đừng dùng MSE** với sigmoid — non-convex và gradient bão hoà. Luôn dùng BCE.
4. Multi-class: dùng **softmax** + cross-entropy.
5. **Threshold** mặc định 0.5 — có thể tune theo bài toán cụ thể (ưu tiên precision hay recall).
6. Logistic Regression đơn giản nhưng cực kỳ mạnh — luôn nên thử trước khi dùng model phức tạp.

# BÀI TẬP VỀ NHÀ

## Bài 1: Polynomial features cho Logistic
Trên `make_moons` (dữ liệu cong không tách tuyến tính được):
1. Train Logistic Regression thường — bao nhiêu accuracy?
2. Thêm `PolynomialFeatures(degree=3)` trước khi fit. Bao nhiêu accuracy?
3. Vẽ decision boundary cho cả hai. Quan sát: Logistic + polynomial cũng học được boundary cong.

## Bài 2: Regularization C
Trên Breast Cancer, sweep `C ∈ {0.001, 0.01, 0.1, 1, 10, 100, 1000}`. (Lưu ý: trong sklearn, $C = 1/\alpha$ — C nhỏ = regularization mạnh.)

1. Vẽ test accuracy theo C (log scale).
2. Vẽ ||w||² theo C.
3. Quan sát: C nhỏ → coef nhỏ, C lớn → coef lớn → có thể overfit.

## Bài 3: L1 vs L2
Train Logistic Regression với `penalty='l2'` và `penalty='l1'` (cần `solver='liblinear'` cho L1). Đếm số coef khác 0 trong mỗi trường hợp. L1 có "chọn feature" giống Lasso không?

## Bài 4: Class imbalance
Sinh dữ liệu 95/5 mất cân bằng. Train Logistic. Báo cáo accuracy, F1 cho lớp ít.

Sau đó dùng `class_weight='balanced'`. So sánh F1 lớp ít trước vs sau.

## Bài 5: So sánh với Linear SVM và KNN
Trên Breast Cancer (đã scale), train 3 model:
1. `LogisticRegression()`
2. `LinearSVC()` hoặc `SVC(kernel='linear')`
3. `KNeighborsClassifier(n_neighbors=7)`

So sánh accuracy + F1. Cái nào ổn nhất? Vì sao Logistic và Linear SVM thường rất gần nhau? *Cả hai đều tìm hyperplane tuyến tính, chỉ khác ở loss function.*